In [ ]:
#importe das bibliotecas utilizadas

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time


In [ ]:
#Processamento de imagem usando open-cv (PRINCIPAL)
#Bruno Vanjura Vieira

def pegar_informacoes(caminho):
    # 1. Abrir o vÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­deo
    video = cv2.VideoCapture(caminho)

    # InformaÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Âµes do vÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­deo
    fps = video.get(cv2.CAP_PROP_FPS)
    total_frames = video.get(cv2.CAP_PROP_FRAME_COUNT)

    duracao = total_frames / fps

    largura = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    pixels_por_frame = largura * altura
    total_pixels_video = pixels_por_frame * total_frames

    print("FPS:", fps)
    print("DuraaÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o:", duracao, "segundos")

    print(f"ResoluÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o: {largura}x{altura}")
    print(f"Pixels por quadro: {pixels_por_frame}")
    print(f"Total de quadros: {total_frames}")
    print(f"Total de pixels no vÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo: {total_pixels_video}")

    video.release()

#FunÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o para obter matriz M:

def obter_matriz_M(caminho_video, largura=160, altura=120):

    # Abrir o vÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­deo
    video = cv2.VideoCapture(caminho_video)

    if not video.isOpened():
        raise Exception("NÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â£o foi possÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­vel abrir o vÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­deo!")

    # Extrair os quadros
    frames = []

    while True:
        ret, frame = video.read()

        if not ret:
            break

        # Converter para escala de cinza
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Redimensionar os quadros
        gray = cv2.resize(gray, (largura, altura))

        frames.append(gray)

    video.release()

    # Verificar se foram extraÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­dos quadros
    if len(frames) == 0:
        raise Exception("Nenhum quadro foi extraÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­do!")

    # Vetorizar cada quadro
    vetores = [frame.flatten() for frame in frames]

    # Construir a matriz M
    M = np.column_stack(vetores).astype(float)

    return M







In [ ]:
# SVD pelo metodo QR aplicado a matriz de Gram M.T @ M.
def DecompsicaoSVD(A, tol=1e-10, max_iter=5000):
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or min(A.shape) == 0:
        raise ValueError("A deve ser uma matriz nao vazia.")
    inicio = time.perf_counter()
    m, n = A.shape
    G = A.T @ A
    G = 0.5 * (G + G.T)
    T = G.copy()
    V = np.eye(n)
    escala = max(np.linalg.norm(T, ord="fro"), 1.0)
    for _ in range(max_iter):
        if np.linalg.norm(T - np.diag(np.diag(T)), ord="fro") <= tol * escala:
            break
        Q, R = np.linalg.qr(T)
        T = R @ Q
        T = 0.5 * (T + T.T)
        V = V @ Q
    else:
        raise RuntimeError("A iteracao QR nao convergiu dentro do limite definido.")
    autovalores = np.diag(T).copy()
    ordem = np.argsort(autovalores)[::-1]
    autovalores = np.maximum(autovalores[ordem], 0.0)
    V = V[:, ordem]
    sigma = np.sqrt(autovalores)
    limite = np.finfo(float).eps * max(m, n) * (sigma[0] if sigma.size else 0.0)
    posto = sigma > limite
    sigma = sigma[posto]
    V = V[:, posto]
    U = (A @ V) / sigma[np.newaxis, :] if sigma.size else np.empty((m, 0))
    return U, np.diag(sigma), V.T, time.perf_counter() - inicio


## Metodo de Jacobi desativado

Nesta versao, a decomposicao SVD utiliza somente o metodo QR implementado acima.

In [ ]:
# Comparacao da implementacao QR com a rotina SVD do NumPy em matriz pequena.
A = np.array([[1, 2, 3], [9, 0, 7], [2, 5, 0]], dtype=float)
u_qr, s_qr, vt_qr, tempo_qr = DecompsicaoSVD(A)
u_np, s_np, vt_np = np.linalg.svd(A, full_matrices=False)
erro_qr = np.linalg.norm(A - u_qr @ s_qr @ vt_qr, ord="fro")
print("Dimensoes de U, S e Vt:", u_qr.shape, s_qr.shape, vt_qr.shape)
print("Erro de reconstrucao QR:", erro_qr)
print("Tempo do metodo QR:", tempo_qr, "segundos")
print("Erro dos valores singulares:", np.linalg.norm(np.diag(s_qr) - s_np))


In [ ]:
# Parte 3: aproximacao de baixo posto.# As funcoes ficam definidas aqui. A decomposicao e executada uma unica vez# na celula principal, que tambem mede a RAM de cada etapa.def aproximacao_baixo_posto(U, S, Vt, k):    """Constroi a aproximacao de posto k sem criar Sigma_k explicitamente."""    s = np.diag(S) if np.ndim(S) == 2 else np.asarray(S)    r = len(s)    if not 1 <= k <= r:        raise ValueError(f"k deve estar entre 1 e {r}.")    return (U[:, :k] * s[:k]) @ Vt[:k, :]def avaliar_aproximacao(s, k):    """Calcula o erro de Frobenius pela cauda dos valores singulares."""    norma_M = np.linalg.norm(s)    erro = np.linalg.norm(s[k:])    return {        "erro_frobenius": erro,        "erro_relativo": erro / norma_M if norma_M else 0.0,    }

In [ ]:
# Visualizacao do quadro e da aproximacao; chamada pela celula principal.def visualizar_aproximacao(M, L_k, k, largura=160, altura=120):    idx_quadro = M.shape[1] // 2    quadro_original = M[:, idx_quadro].reshape((altura, largura))    quadro_Lk = np.clip(L_k[:, idx_quadro].reshape((altura, largura)), 0, 255)    quadro_residuo = np.abs(M[:, idx_quadro] - L_k[:, idx_quadro]).reshape((altura, largura))    fig, axes = plt.subplots(1, 3, figsize=(15, 4))    axes[0].imshow(quadro_original, cmap="gray", vmin=0, vmax=255)    axes[0].set_title(f"Original (quadro {idx_quadro})")    axes[0].axis("off")    axes[1].imshow(quadro_Lk, cmap="gray", vmin=0, vmax=255)    axes[1].set_title(f"Aproximacao L_k (k={k})")    axes[1].axis("off")    axes[2].imshow(quadro_residuo, cmap="gray")    axes[2].set_title(f"Residuo |M - L_k| (k={k})")    axes[2].axis("off")    plt.tight_layout()    plt.show()

In [ ]:
# Parte 4: IdentificaÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o de Movimento
# Caio Lucas IndalÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â©cio

def mat_res(M, L_k):
    R_k = M - L_k # Dada uma matriz e um L_k, calcula-se o resÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­duo R_k (movimento)
    S_k = np.abs(R_k) # S_k ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â© uma matriz cujos elementos sÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o o mÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â³dulo dos elementos de R_k
    return R_k, S_k

# Criando uma mascara binÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¡ria
def mask(S_k):
    threshold = 50
    # Mask ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â© uma matriz que transforma os valores acima de um limiar em 1 e abaixo ou igual em zero
    # Mask ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â©, portanto, uma mÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¡scara binÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¡ria.
    mask = (S_k > threshold).astype(np.uint8)
    return mask

# Nota-se que quanto maior o threshold

In [ ]:
# Parte 5: ReconstruÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â§ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o do vÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo# Luiz Gustavo Borges e Cunha - Rizzioli nÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o fez nadadef recontruir_video(M, L_k, S_k, caminho_original, caminho_saida,                     largura_matriz=160, altura_matriz=120):    """Gera um vÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo lado a lado: original, fundo aproximado e movimento."""    captura = cv2.VideoCapture(caminho_original)    if not captura.isOpened():        raise ValueError("NÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o foi possÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­vel abrir o vÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo original.")    largura = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))    altura = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))    fps = captura.get(cv2.CAP_PROP_FPS)    if L_k.shape != M.shape or S_k.shape != M.shape:        captura.release()        raise ValueError("M, L_k e S_k devem ter as mesmas dimensÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Âµes.")    codec = cv2.VideoWriter_fourcc(*"mp4v")    video = cv2.VideoWriter(caminho_saida, codec, fps, (3 * largura, altura))    if not video.isOpened():        captura.release()        raise ValueError("NÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â£o foi possÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­vel criar o vÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo de saÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­da.")    try:        for i in range(M.shape[1]):            ret, original = captura.read()            if not ret:                raise ValueError("O vÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo original tem menos quadros que M.")            fundo = L_k[:, i].reshape((altura_matriz, largura_matriz))            fundo = np.clip(fundo, 0, 255).astype(np.uint8)            fundo = cv2.resize(fundo, (largura, altura))            fundo = cv2.cvtColor(fundo, cv2.COLOR_GRAY2BGR)            movimento = S_k[:, i].reshape((altura_matriz, largura_matriz))            mascara = mask(movimento) * 255            mascara = cv2.resize(                mascara, (largura, altura), interpolation=cv2.INTER_NEAREST            )            regioes_movimento = cv2.bitwise_and(                original, original, mask=mascara            )            # Cada quadro de saÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­da reÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Âºne os trÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Âªs painÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â©is na ordem do projeto.            video.write(np.hstack((original, fundo, regioes_movimento)))    finally:        captura.release()        video.release()    print(f"VÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â­deo comparativo salvo em: {caminho_saida}")

In [ ]:
# Codigo principal: mesmo fluxo da main.ipynb, usando QR no lugar de Jacobi.# O monitor amostra a RAM (RSS) do processo a cada 20 ms em cada etapa.import osimport threadingfrom contextlib import contextmanagertry:    import psutilexcept ImportError as exc:    raise ImportError(        "Para rastrear a RAM, instale psutil no ambiente do notebook: pip install psutil"    ) from exc_processo = psutil.Process(os.getpid())@contextmanagerdef medir_ram(nome_etapa):    rss_inicio = _processo.memory_info().rss    pico = [rss_inicio]    parar = threading.Event()    def amostrar():        while not parar.is_set():            try:                pico[0] = max(pico[0], _processo.memory_info().rss)            except psutil.Error:                pass            parar.wait(0.02)    monitor = threading.Thread(target=amostrar, daemon=True)    monitor.start()    inicio = time.perf_counter()    try:        yield    finally:        duracao = time.perf_counter() - inicio        parar.set()        monitor.join()        rss_fim = _processo.memory_info().rss        pico[0] = max(pico[0], rss_fim)        print(            f"{nome_etapa}: {duracao:.2f} s | "            f"RAM inicio {rss_inicio / 1024**2:.1f} MiB | "            f"pico {pico[0] / 1024**2:.1f} MiB "            f"(+{max(0, pico[0] - rss_inicio) / 1024**2:.1f} MiB) | "            f"fim {rss_fim / 1024**2:.1f} MiB"        )largura, altura = 160, 120pegar_informacoes("data/caio_lutando.mp4")with medir_ram("Leitura do video e construcao de M"):    M = obter_matriz_M("data/caio_lutando.mp4", largura=largura, altura=altura)# Usa QR implementado no notebook no lugar da decomposicao de Jacobi.with medir_ram("Decomposicao SVD pelo metodo QR"):    U_r, Sigma_r, Vt_r, tempo_qr = DecompsicaoSVD(M)print("Dimensoes de U, S e Vt:", U_r.shape, Sigma_r.shape, Vt_r.shape)print("Tempo reportado pela funcao QR:", tempo_qr, "segundos")# Mostra os erros de baixo posto sem construir uma matriz para cada k.s_valores = np.diag(Sigma_r)valores_k = [k for k in [1, 2, 3, 4, 5, 10, 20, 100] if k <= len(s_valores)]for k in valores_k:    metricas = avaliar_aproximacao(s_valores, k)    print(        f"k = {k:3d} | erro Frobenius = {metricas['erro_frobenius']:.4f} | "        f"erro relativo = {metricas['erro_relativo'] * 100:.2f}%"    )with medir_ram("Aproximacao de baixo posto (k=1)"):    L_k = aproximacao_baixo_posto(U_r, Sigma_r, Vt_r, 1)with medir_ram("Identificacao do movimento (residuo)"):    R_k, S_k = mat_res(M, L_k)with medir_ram("Reconstrucao e gravacao do video"):    recontruir_video(        M, L_k, S_k,        "data/caio_lutando.mp4",        "data/video_reconstruido.mp4",        largura_matriz=largura,        altura_matriz=altura,    )visualizar_aproximacao(M, L_k, 1, largura=largura, altura=altura)